In [ ]:
import gym
import matplotlib.pyplot as plt
import numpy as np
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.distributions as distributions

#optional
import sys
import pickle
import time

In [ ]:
# Create the environment
env = gym.make("Acrobot-v1")

# Set up matplotlib for interactive plotting
# (for use in Jupyter notebooks or IPython)
is_ipython = "inline" in plt.get_backend()
if is_ipython:
    from IPython import display
plt.ion()

# Set up the device to use (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print the device being used
print(device)

In [ ]:
class Network(nn.Module):
    def __init__(self, n_observation,n_actions):
        super(Network, self).__init__()
        self.dense1 = nn.Linear(n_observations,32)
        self.out = nn.Linear(32,n_actions)

    def forward(self, x):
        x = F.relu(self.dense1(x))
        logits = self.out(x)
        dist = distributions.Categorical(logits=logits)
        action = dist.sample()
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        return logits, action, probs, log_probs

In [ ]:
def train_step(batch_states, batch_actions, batch_returns):
    # Zero gradients
    optimizer.zero_grad()
    # Forward pass
    batch_states_tensor = torch.cat(batch_states, dim=0)
    logits, actions, probs, log_probs = net(batch_states_tensor)
    
    # One-hot encode actions
    action_masks = torch.eye(n_actions)[batch_actions]
    
    # Masked log probabilities
    masked_log_probs = torch.sum(action_masks * log_probs, dim=-1)
    batch_returns_tensor = torch.tensor(batch_returns)
    # Compute loss
    loss = -torch.mean(batch_returns_tensor * masked_log_probs)
    
    # Backward pass
    loss.backward()
    
    # Update weights
    optimizer.step()
    
    return loss.item()

In [ ]:
n_actions = env.action_space.n
state, _ = env.reset()
n_observations = len(state)

net = Network(n_observations, n_actions)
optimizer = optim.Adam(net.parameters(), lr=2e-2)
start_time = time.time()

steps_per_train_step = 1000
last_100_ep_ret= []
num_episodes = 2000
eps = 1
batch_states, batch_actions, batch_returns = [], [], []
total_rewards = []
for episode in range(num_episodes):
    state, _ = env.reset()
    done, ep_rew = False, []
    count =0 
    while not done and count<7000:
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        logits, action, probs, log_probs = net(state)
        action = action[0].item()
        next_state, reward, done, _ , _= env.step(action)
        batch_states.append(state)
        batch_actions.append(action)
        ep_rew.append(reward)
        #if done==1:
        #    print(f"Episode: {episode} done, {len(batch_states)} steps")
        count+=1
        state = next_state
    episode_ret = sum(ep_rew)
    total_rewards.append(episode_ret)
    episode_len = len(ep_rew)
    batch_returns += [episode_ret] * episode_len
    if len(batch_states) >= steps_per_train_step:
    # Now that we have enough experience for this policy, train it on-policy.
        loss = train_step(batch_states, batch_actions,batch_returns)
    # Print the performance of the policy.
        print(f"Episode: {episode}, Loss: {loss:.2f}, Retrurn: {np.mean(batch_returns):.2f}")
        batch_states, batch_actions, batch_returns = [], [], []

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")

torch.save(net.state_dict(),'pg_state_dict.pth')

# Plotting the total rewards per episode
window_size = 20 # adjust as needed
total_rewards_smooth = np.convolve(total_rewards, np.ones(window_size)/window_size, mode='valid')

# Plotting the smoothed total rewards per episode
plt.plot(total_rewards_smooth)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Mean Reward last 20 episodes')
plt.savefig('Policy gradients training.png')
plt.show()

In [ ]:
#Evaluation
env = gym.make("Acrobot-v1")

ep_rewards = []
for episode in range(500):
    state, _ = env.reset()
    done = False
    ep_reward = 0
    while not done:
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        logits, action, probs, log_probs = net(state)
        action = action[0].item()
        #energy +=
        # Step the environment
        state, reward, done, _ , _= env.step(action)
        ep_reward += reward
    ep_rewards.append(ep_reward)
    #print(f"episode: {episode}, rewards: {ep_reward}")

mean = sum(ep_rewards) / len(ep_rewards)
#plt.plot(ep_rewards)
plt.scatter(range(len(ep_rewards)), ep_rewards, marker='o')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Policy gradients evaluation, 500 episode rewards')
plt.savefig('policy gradients evaluation.png')
plt.show()

print(mea